# EDA - Gym Churn Predictor

Este notebook documenta a analise exploratoria da base `gym_churn_us.csv` para a entrega da Semana 5 do projeto de predicao de churn.

Objetivos:

- carregar a base de clientes;
- verificar qualidade dos dados;
- limpar duplicados e nulos;
- visualizar a distribuicao de churn;
- comparar clientes que ficaram versus clientes que cancelaram;
- justificar as features usadas no modelo preditivo.

## 1. Importacao de bibliotecas e configuracao

O notebook usa `pandas` para manipulacao dos dados e `matplotlib` para graficos simples. A base deve estar em `dataset/gym_churn_us.csv` na raiz do projeto.

In [ ]:
import numpy as np

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("default")
pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../dataset/gym_churn_us.csv")
DATA_PATH

## 2. Carga da base

A primeira etapa da EDA e carregar o arquivo CSV e conferir dimensoes, colunas e primeiras linhas.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Linhas: {df.shape[0]:,}".replace(",", "."))
print(f"Colunas: {df.shape[1]}")
df.head()

In [ ]:
df.info()

## 3. Qualidade dos dados

Aqui verificamos valores nulos, duplicados e a distribuicao da variavel alvo `Churn`.

In [ ]:
quality_summary = pd.DataFrame({
    "nulos": df.isna().sum(),
    "tipo": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
})

print(f"Duplicados: {df.duplicated().sum()}")
quality_summary

In [ ]:
df_clean = df.drop_duplicates().dropna().copy()

print(f"Shape original: {df.shape}")
print(f"Shape apos limpeza: {df_clean.shape}")
print(f"Linhas removidas: {len(df) - len(df_clean)}")

## 4. Distribuicao do churn

A variavel `Churn` indica se o cliente cancelou (`1`) ou permaneceu (`0`).

In [ ]:
churn_counts = df_clean["Churn"].value_counts().sort_index()
churn_rate = df_clean["Churn"].mean()

print(churn_counts)
print(f"Taxa de churn: {churn_rate:.2%}")

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Ficou (0)", "Cancelou (1)"]
ax.bar(labels, churn_counts.values, color=["#2563eb", "#dc2626"])
ax.set_title("Distribuicao de clientes por churn")
ax.set_ylabel("Quantidade de clientes")
ax.bar_label(ax.containers[0])
plt.show()

## 5. Estatisticas descritivas

As estatisticas ajudam a entender escala, dispersao e comportamento das variaveis numericas.

In [ ]:
df_clean.describe().T

## 6. Comparacao entre clientes que ficaram e cancelaram

A comparacao por grupos mostra quais variaveis mudam mais entre clientes com `Churn = 0` e `Churn = 1`.

In [ ]:
features_comparacao = [
    "Lifetime",
    "Avg_class_frequency_current_month",
    "Age",
    "Contract_period",
    "Month_to_end_contract",
    "Avg_class_frequency_total",
    "Avg_additional_charges_total",
]

group_means = df_clean.groupby("Churn")[features_comparacao].mean().T
group_means.columns = ["Ficou (0)", "Cancelou (1)"]
group_means.round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

plot_features = [
    "Lifetime",
    "Avg_class_frequency_current_month",
    "Contract_period",
    "Avg_additional_charges_total",
]

for ax, feature in zip(axes, plot_features):
    df_clean.boxplot(column=feature, by="Churn", ax=ax)
    ax.set_title(feature)
    ax.set_xlabel("Churn")
    ax.set_ylabel("Valor")

plt.suptitle("Comparacao de variaveis por churn")
plt.tight_layout()
plt.show()

## 7. Variaveis categoricas/binarias

As variaveis binarias ajudam a observar diferencas de comportamento em grupos como aulas em grupo, indicacao e proximidade da academia.

In [ ]:
binary_features = ["Group_visits", "Promo_friends", "Partner", "Near_Location"]

binary_churn = []
for feature in binary_features:
    rates = df_clean.groupby(feature)["Churn"].mean().rename("churn_rate")
    for value, rate in rates.items():
        binary_churn.append({"feature": feature, "valor": value, "churn_rate": rate})

pd.DataFrame(binary_churn).round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, feature in zip(axes, binary_features):
    rates = df_clean.groupby(feature)["Churn"].mean().sort_index()
    ax.bar(rates.index.astype(str), rates.values, color="#2563eb")
    ax.set_title(f"Taxa de churn por {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("Taxa de churn")
    ax.set_ylim(0, max(rates.max() * 1.2, 0.1))
    ax.bar_label(ax.containers[0], fmt="%.2f")

plt.tight_layout()
plt.show()

## 8. Correlacao com churn e selecao de features

A correlacao e usada aqui como uma analise inicial para orientar a selecao de features. Variaveis com correlacao proxima de zero tendem a contribuir pouco nesta primeira versao.

In [ ]:
corr = df_clean.corr(numeric_only=True)["Churn"].sort_values(key=lambda s: s.abs(), ascending=False)
corr_df = corr.drop("Churn").to_frame("correlacao_com_churn")
corr_df

In [ ]:
top_corr = corr_df.head(12).sort_values("correlacao_com_churn")

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#dc2626" if value > 0 else "#2563eb" for value in top_corr["correlacao_com_churn"]]
ax.barh(top_corr.index, top_corr["correlacao_com_churn"], color=colors)
ax.set_title("Correlacao das variaveis com churn")
ax.set_xlabel("Correlacao")
plt.tight_layout()
plt.show()

In [ ]:
features_modelo = [
    "Lifetime",
    "Avg_class_frequency_current_month",
    "Age",
    "Contract_period",
    "Month_to_end_contract",
    "Avg_class_frequency_total",
    "Avg_additional_charges_total",
    "Group_visits",
    "Promo_friends",
    "Partner",
    "Near_Location",
]

descartadas = [col for col in ["gender", "Phone"] if col in df_clean.columns]

print("Features usadas no modelo:")
print(features_modelo)
print("\nFeatures descartadas por baixa relacao inicial com churn:")
print(descartadas)

## 9. Conclusoes da EDA

- A base permite treinar uma primeira versao funcional de modelo de churn.
- `Churn` e a variavel alvo e representa clientes que cancelaram.
- Variaveis como tempo como cliente, frequencia recente, contrato e meses ate o vencimento aparecem como sinais relevantes.
- `gender` e `Phone` foram deixadas fora desta primeira versao por baixa relacao inicial com churn.
- A EDA suporta a arquitetura da Semana 5: upload do CSV, treino do RandomForest no backend e inferencia pela interface web.

## 10. Features derivadas do case

O documento de entregáveis pede variáveis derivadas para capturar queda de engajamento e segmentos de risco. As três principais são:

- `ratio_freq_atual_vs_lifetime`: compara frequência atual com frequência histórica;
- `flag_early_user`: marca clientes com até 1 mês de relacionamento;
- `flag_sleeping_dog`: marca clientes antigos com frequência atual muito baixa.

In [ ]:
df_eda = df_clean.copy()

# Evita divisão por zero quando o histórico de frequência for igual a 0.
df_eda["ratio_freq_atual_vs_lifetime"] = (
    df_eda["Avg_class_frequency_current_month"]
    / df_eda["Avg_class_frequency_total"].replace(0, np.nan)
).fillna(0)

df_eda["flag_early_user"] = (df_eda["Lifetime"] <= 1).astype(int)
df_eda["flag_sleeping_dog"] = (
    (df_eda["Lifetime"] >= 6)
    & (df_eda["Avg_class_frequency_current_month"] < 1)
).astype(int)

df_eda[[
    "ratio_freq_atual_vs_lifetime",
    "flag_early_user",
    "flag_sleeping_dog",
    "Churn",
]].groupby("Churn").mean().round(3)

## 11. Visualizações obrigatórias

As visualizações abaixo reforçam a leitura executiva da base: proporção de churn, comparação por tempo de contrato, queda de frequência, perfil por idade/lifetime e mapa de correlação com as variáveis originais e derivadas.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Visualizações obrigatórias da EDA de churn", fontsize=16, fontweight="bold")

churn_counts = df_eda["Churn"].value_counts().sort_index()
axes[0, 0].bar(["Ficou", "Cancelou"], churn_counts.values, color=["#2563eb", "#dc2626"])
axes[0, 0].set_title("Distribuição de churn")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Clientes")

contract_churn = df_eda.groupby("Contract_period")["Churn"].mean().sort_index()
axes[0, 1].bar(contract_churn.index.astype(str), contract_churn.values, color="#2563eb")
axes[0, 1].set_title("Taxa de churn por duração de contrato")
axes[0, 1].set_xlabel("Contrato (meses)")
axes[0, 1].set_ylabel("Taxa média de churn")

box_data = [
    df_eda.loc[df_eda["Churn"] == 0, "Avg_class_frequency_current_month"],
    df_eda.loc[df_eda["Churn"] == 1, "Avg_class_frequency_current_month"],
]
axes[1, 0].boxplot(box_data, labels=["Ficou", "Cancelou"], patch_artist=True)
axes[1, 0].set_title("Frequência atual por status de churn")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("Aulas/semana no mês")

for churn_value, color, label in [(0, "#2563eb", "Ficou"), (1, "#dc2626", "Cancelou")]:
    subset = df_eda[df_eda["Churn"] == churn_value]
    axes[1, 1].scatter(subset["Age"], subset["Lifetime"], alpha=0.6, color=color, label=label)
axes[1, 1].set_title("Idade x tempo como cliente")
axes[1, 1].set_xlabel("Idade")
axes[1, 1].set_ylabel("Lifetime (meses)")
axes[1, 1].legend(title="Churn")

plt.tight_layout()
plt.show()

In [ ]:
features_para_correlacao = FEATURES + [
    "ratio_freq_atual_vs_lifetime",
    "flag_early_user",
    "flag_sleeping_dog",
    "Churn",
]

corr_matrix = df_eda[features_para_correlacao].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(12, 8))
heatmap = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index)
ax.set_title("Mapa de correlação entre features e churn")
fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

correlacao_churn = (
    corr_matrix["Churn"]
    .drop("Churn")
    .sort_values(key=abs, ascending=False)
)

correlacao_churn.head(10).to_frame("correlacao_com_churn").round(3)

In [ ]:
segmentos = pd.DataFrame({
    "segmento": [
        "Usuários novos",
        "Sleeping dogs",
        "Contrato curto",
        "Baixa frequência atual",
        "Sem visitas em grupo",
    ],
    "clientes": [
        int(df_eda["flag_early_user"].sum()),
        int(df_eda["flag_sleeping_dog"].sum()),
        int((df_eda["Contract_period"] <= 1).sum()),
        int((df_eda["Avg_class_frequency_current_month"] < 1).sum()),
        int((df_eda["Group_visits"] == 0).sum()),
    ],
    "churn_rate": [
        df_eda.loc[df_eda["flag_early_user"] == 1, "Churn"].mean(),
        df_eda.loc[df_eda["flag_sleeping_dog"] == 1, "Churn"].mean(),
        df_eda.loc[df_eda["Contract_period"] <= 1, "Churn"].mean(),
        df_eda.loc[df_eda["Avg_class_frequency_current_month"] < 1, "Churn"].mean(),
        df_eda.loc[df_eda["Group_visits"] == 0, "Churn"].mean(),
    ],
})
segmentos["churn_rate"] = segmentos["churn_rate"].fillna(0).round(3)
segmentos.sort_values("churn_rate", ascending=False)

## 12. Leitura executiva das features derivadas

As features derivadas ajudam a traduzir comportamento em ação:

- queda na frequência recente indica perda de hábito e deve acionar contato preventivo;
- clientes no início do relacionamento precisam de onboarding ativo;
- clientes antigos com baixa frequência são bons candidatos a campanhas de reativação;
- contratos curtos concentram maior vulnerabilidade e merecem incentivo para migração de plano;
- ausência de aulas em grupo sugere menor vínculo social com a academia.